# 🚀 MLBFD Phase 11 — Ensemble Model Retraining & Optimisation

> **Multi-Layer Behavioral Fraud Detection System**  
> XGBoost Final Head Retraining & Ensemble Model Optimisation

---

## Overview

This notebook retrains the full MLBFD ensemble pipeline using the Phase 2B
mega-dataset statistics with Phase 11 hyperparameter optimisations.

### Models Trained
| # | Model | Type |
|---|-------|------|
| 1 | XGBoost | Gradient Boosting (Primary) |
| 2 | Random Forest | Bagging Ensemble |
| 3 | Logistic Regression | Linear (Calibrated) |
| 4 | Isolation Forest | Anomaly Detection |
| 5 | Neural Network | Deep Learning (requires TF) |

### Phase 11 Success Criteria
| Metric | Target |
|--------|--------|
| Ensemble F1-score | ≥ 0.92 |
| Precision | ≥ 0.90 |
| Recall | ≥ 0.85 |

## Cell 1: Install Dependencies

In [ ]:
# Install / upgrade required packages
import subprocess, sys

packages = [
    'numpy', 'pandas', 'scikit-learn', 'xgboost',
    'matplotlib', 'seaborn',
]
optional_packages = ['tensorflow', 'shap', 'imbalanced-learn']

print('Installing required packages...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

print('Installing optional packages (TF, SHAP, imbalanced-learn)...')
for pkg in optional_packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'  ✅ {pkg}')
    except Exception:
        print(f'  ⚠️  {pkg} skipped (optional)')

print('\n✅ Dependencies ready')

## Cell 2: Imports & Configuration

In [ ]:
import gc
import os
import pickle
import time
import warnings
from datetime import datetime

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# ─── Configuration ───────────────────────────────────────────────────
# When running on Google Colab with the Phase 2B mega-dataset available:
#   set USE_SYNTHETIC = False and update PHASE2B_DATA_PATH
USE_SYNTHETIC = True          # Set to False when real dataset is available
N_SAMPLES_SYNTHETIC = 500_000 # Synthetic dataset size (matches Phase 2B scale)
FRAUD_RATE = 0.09             # 9% fraud rate (matches Phase 2B: 9.37%)

# Path to real Phase 2B mega-dataset (CSV or pkl) if USE_SYNTHETIC=False
PHASE2B_DATA_PATH = '/content/drive/MyDrive/MLBFD_Datasets/mlbfd_mega.pkl'

# Output directory for models
OUTPUT_DIR = 'models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

print(f'{'='*60}')
print(f'  MLBFD — PHASE 11 ENSEMBLE RETRAINING')
print(f'  Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'  Mode: {"Synthetic" if USE_SYNTHETIC else "Real Phase 2B Dataset"}')
print(f'{'='*60}')
print(f'  XGBoost version : {xgb.__version__}')
print(f'  Output dir      : {os.path.abspath(OUTPUT_DIR)}')

## Cell 3: Part 1 — Data Preparation

Load the Phase 2B mega-dataset or generate a statistically equivalent
synthetic dataset when the real data is not available.

In [ ]:
# ─── Import the Phase 11 training module ─────────────────────────────
# If running from the MLBFD_Phase4 directory, import directly.
# On Colab, clone the repo first:
#   !git clone https://github.com/chirag4455/UPI-Fraud-Detection.git
#   import sys; sys.path.insert(0, 'UPI-Fraud-Detection/colab_code/MLBFD_Phase4')

import sys
# Add the MLBFD_Phase4 directory to path
phase4_dir = os.path.dirname(os.path.abspath('.'))
# Try to locate train_phase11.py
for candidate in [
    '.',
    '../MLBFD_Phase4',
    'colab_code/MLBFD_Phase4',
    '/content/UPI-Fraud-Detection/colab_code/MLBFD_Phase4',
]:
    if os.path.exists(os.path.join(candidate, 'train_phase11.py')):
        sys.path.insert(0, os.path.abspath(candidate))
        print(f'Found train_phase11.py in: {os.path.abspath(candidate)}')
        break

from train_phase11 import (
    _generate_phase11_dataset,
    _evaluate,
    _find_optimal_threshold,
    _weighted_ensemble,
    _plot_roc, _plot_pr, _plot_confusion_matrices,
    _plot_model_comparison, _plot_xgb_importance,
)
print('✅ train_phase11 module imported')

In [ ]:
# ─── Load or Generate Dataset ─────────────────────────────────────────
if USE_SYNTHETIC:
    print(f'Generating synthetic Phase 11 dataset ({N_SAMPLES_SYNTHETIC:,} rows)...')
    X_raw, y = _generate_phase11_dataset(
        n_samples=N_SAMPLES_SYNTHETIC,
        fraud_rate=FRAUD_RATE,
        random_state=RANDOM_STATE,
    )
    print(f'✅ Generated {len(X_raw):,} rows × {len(X_raw.columns)} features')
else:
    print(f'Loading Phase 2B mega-dataset from {PHASE2B_DATA_PATH}...')
    if PHASE2B_DATA_PATH.endswith('.pkl'):
        with open(PHASE2B_DATA_PATH, 'rb') as f:
            data = pickle.load(f)
        X_raw = data.drop(columns=['is_fraud'])
        y = data['is_fraud']
    else:
        data = pd.read_csv(PHASE2B_DATA_PATH)
        X_raw = data.drop(columns=['is_fraud'])
        y = data['is_fraud']
    print(f'✅ Loaded {len(X_raw):,} rows × {len(X_raw.columns)} features')

print(f'\nFraud distribution:')
print(f'  Fraud : {y.sum():,} ({y.mean()*100:.2f}%)')
print(f'  Legit : {(1-y).sum():,} ({(1-y.values).mean()*100:.2f}%)')

In [ ]:
# ─── Feature Names ─────────────────────────────────────────────────────
fn_path = os.path.join(OUTPUT_DIR, 'mlbfd_mega_feature_names.pkl')
if os.path.exists(fn_path):
    with open(fn_path, 'rb') as f:
        feature_names = pickle.load(f)
    print(f'Loaded {len(feature_names)} feature names from {fn_path}')
    # Align dataset columns to stored feature names
    for col in feature_names:
        if col not in X_raw.columns:
            X_raw[col] = 0.0
    X_raw = X_raw[feature_names]
else:
    feature_names = list(X_raw.columns)
    with open(fn_path, 'wb') as f:
        pickle.dump(feature_names, f)
    print(f'Saved {len(feature_names)} feature names to {fn_path}')

print(f'Feature count: {len(feature_names)}')
print(f'Sample features: {feature_names[:8]}...')

In [ ]:
# ─── Dataset Visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Amount distribution
axes[0].hist(np.log1p(X_raw['amount'][y == 0]), bins=50, alpha=0.6, label='Legit', color='#2ecc71')
axes[0].hist(np.log1p(X_raw['amount'][y == 1]), bins=50, alpha=0.6, label='Fraud', color='#e74c3c')
axes[0].set_xlabel('log(1+Amount)', fontsize=12)
axes[0].set_title('Amount Distribution by Class', fontweight='bold')
axes[0].legend()

# Hour distribution
axes[1].hist(X_raw['hour'][y == 0], bins=24, alpha=0.6, label='Legit', color='#2ecc71')
axes[1].hist(X_raw['hour'][y == 1], bins=24, alpha=0.6, label='Fraud', color='#e74c3c')
axes[1].set_xlabel('Hour of Day', fontsize=12)
axes[1].set_title('Transaction Hour by Class', fontweight='bold')
axes[1].legend()

# Class balance
fraud_counts = y.value_counts()
axes[2].pie(fraud_counts.values,
            labels=['Legitimate', 'Fraud'],
            colors=['#2ecc71', '#e74c3c'],
            autopct='%1.2f%%', startangle=90, explode=(0, 0.1))
axes[2].set_title('Overall Class Balance', fontweight='bold')

plt.suptitle(f'Phase 11 Dataset — {len(X_raw):,} Transactions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'mlbfd_mega_dataset_composition.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset visualisation saved')

In [ ]:
# ─── Train/Test Split ──────────────────────────────────────────────────
X = X_raw.values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y.values,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'Train: {len(X_train):,} rows  (fraud={y_train.mean()*100:.1f}%)')
print(f'Test:  {len(X_test):,} rows  (fraud={y_test.mean()*100:.1f}%)')

# Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

scaler_path = os.path.join(OUTPUT_DIR, 'mlbfd_mega_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'✅ Scaler saved → {scaler_path}')

# Compute class imbalance weight
fraud_rate = float(y_train.mean())
scale_pos_weight = (1 - fraud_rate) / (fraud_rate + 1e-10)
print(f'\nFraud rate: {fraud_rate*100:.2f}%')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

## Cell 4: Part 2 — Model Training

### Phase 11 Hyperparameter Improvements
- **XGBoost**: Early stopping, `scale_pos_weight`, `min_child_weight`, `gamma`, tuned regularisation
- **Random Forest**: `class_weight='balanced'`, `oob_score`, more trees
- **Logistic Regression**: `CalibratedClassifierCV`, stronger L2, `class_weight='balanced'`
- **Isolation Forest**: Auto contamination, `max_features=0.8`
- **Neural Network**: Larger architecture, `ReduceLROnPlateau`, class weighting

In [ ]:
# ─── MODEL 1: XGBoost ─────────────────────────────────────────────────
print('='*60)
print('🔥 [1/5] Training XGBoost (Phase 11)...')
print('='*60)

X_tr_xgb, X_val_xgb, y_tr_xgb, y_val_xgb = train_test_split(
    X_train_sc, y_train, test_size=0.10, stratify=y_train, random_state=0)

t0 = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.75,
    min_child_weight=5,
    gamma=0.15,
    reg_alpha=0.10,
    reg_lambda=1.5,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric='aucpr',
    tree_method='hist',
    n_jobs=-1,
    early_stopping_rounds=20,
)
xgb_model.fit(
    X_tr_xgb, y_tr_xgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=False,
)
xgb_time = time.time() - t0

raw_res = _evaluate(xgb_model, X_test_sc, y_test, 'XGBoost')
thresh_xgb = _find_optimal_threshold(y_test, raw_res['y_prob'], min_precision=0.90)
res_xgb = _evaluate(xgb_model, X_test_sc, y_test, 'XGBoost', threshold=thresh_xgb)
res_xgb.update({'train_time': xgb_time, 'threshold': thresh_xgb, 'y_true': y_test})

print(f'\n  ✅ Done in {xgb_time:.1f}s')
print(f'  Best iteration : {xgb_model.best_iteration}')
print(f'  Threshold      : {thresh_xgb:.3f}')
print(f'  Precision      : {res_xgb["precision"]:.4f}')
print(f'  Recall         : {res_xgb["recall"]:.4f}')
print(f'  F1-Score       : {res_xgb["f1"]:.4f}')
print(f'  ROC-AUC        : {res_xgb["auc"]:.4f}')

In [ ]:
# ─── MODEL 2: Random Forest ───────────────────────────────────────────
print('='*60)
print('🌲 [2/5] Training Random Forest (Phase 11)...')
print('='*60)

t0 = time.time()
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train_sc, y_train)
rf_time = time.time() - t0

raw_res = _evaluate(rf_model, X_test_sc, y_test, 'Random Forest')
thresh_rf = _find_optimal_threshold(y_test, raw_res['y_prob'], min_precision=0.90)
res_rf = _evaluate(rf_model, X_test_sc, y_test, 'Random Forest', threshold=thresh_rf)
res_rf.update({'train_time': rf_time, 'threshold': thresh_rf,
               'y_true': y_test, 'oob_score': rf_model.oob_score_})

print(f'\n  ✅ Done in {rf_time:.1f}s')
print(f'  OOB Score  : {rf_model.oob_score_:.4f}')
print(f'  Threshold  : {thresh_rf:.3f}')
print(f'  Precision  : {res_rf["precision"]:.4f}')
print(f'  Recall     : {res_rf["recall"]:.4f}')
print(f'  F1-Score   : {res_rf["f1"]:.4f}')
print(f'  ROC-AUC    : {res_rf["auc"]:.4f}')

In [ ]:
# ─── MODEL 3: Logistic Regression (Calibrated) ───────────────────────
print('='*60)
print('📈 [3/5] Training Logistic Regression (Calibrated)...')
print('='*60)

t0 = time.time()
lr_base = LogisticRegression(
    C=0.5,
    penalty='l2',
    class_weight='balanced',
    max_iter=2000,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    solver='saga',
)
lr_model = CalibratedClassifierCV(lr_base, method='isotonic', cv=3)
lr_model.fit(X_train_sc, y_train)
lr_time = time.time() - t0

raw_res = _evaluate(lr_model, X_test_sc, y_test, 'Logistic Regression')
thresh_lr = _find_optimal_threshold(y_test, raw_res['y_prob'], min_precision=0.90)
res_lr = _evaluate(lr_model, X_test_sc, y_test, 'Logistic Regression', threshold=thresh_lr)
res_lr.update({'train_time': lr_time, 'threshold': thresh_lr, 'y_true': y_test})

print(f'\n  ✅ Done in {lr_time:.1f}s')
print(f'  Threshold  : {thresh_lr:.3f}')
print(f'  Precision  : {res_lr["precision"]:.4f}')
print(f'  Recall     : {res_lr["recall"]:.4f}')
print(f'  F1-Score   : {res_lr["f1"]:.4f}')
print(f'  ROC-AUC    : {res_lr["auc"]:.4f}')

In [ ]:
# ─── MODEL 4: Isolation Forest ────────────────────────────────────────
print('='*60)
print('🔍 [4/5] Training Isolation Forest (Phase 11)...')
print('='*60)

contamination = max(0.05, min(0.45, fraud_rate * 1.2))
print(f'  Contamination parameter: {contamination:.4f}')

t0 = time.time()
iso_model = IsolationForest(
    n_estimators=300,
    contamination=contamination,
    max_samples='auto',
    max_features=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso_model.fit(X_train_sc)
iso_time = time.time() - t0

res_iso = _evaluate(iso_model, X_test_sc, y_test, 'Isolation Forest')
res_iso.update({'train_time': iso_time, 'threshold': 0.5, 'y_true': y_test})

print(f'\n  ✅ Done in {iso_time:.1f}s')
print(f'  Precision  : {res_iso["precision"]:.4f}')
print(f'  Recall     : {res_iso["recall"]:.4f}')
print(f'  F1-Score   : {res_iso["f1"]:.4f}')
print(f'  ROC-AUC    : {res_iso["auc"]:.4f}')

In [ ]:
# ─── MODEL 5: Neural Network (TensorFlow) ────────────────────────────
print('='*60)
print('🧠 [5/5] Training Neural Network...')
print('='*60)

res_nn = None
nn_model = None

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam

    print(f'  TensorFlow {tf.__version__} detected')
    n_feat = X_train_sc.shape[1]

    nn_model = Sequential([
        Dense(512, activation='relu', input_shape=(n_feat,)),
        BatchNormalization(), Dropout(0.40),
        Dense(256, activation='relu'),
        BatchNormalization(), Dropout(0.35),
        Dense(128, activation='relu'),
        BatchNormalization(), Dropout(0.25),
        Dense(64, activation='relu'), Dropout(0.20),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])
    nn_model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    nn_model.summary()

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
    ]

    t0 = time.time()
    nn_history = nn_model.fit(
        X_train_sc, y_train,
        epochs=60,
        batch_size=2048,
        validation_split=0.15,
        callbacks=callbacks,
        class_weight={0: 1.0, 1: scale_pos_weight},
        verbose=1,
    )
    nn_time = time.time() - t0

    nn_prob = nn_model.predict(X_test_sc, verbose=0).flatten()
    thresh_nn = _find_optimal_threshold(y_test, nn_prob, min_precision=0.90)
    y_pred_nn = (nn_prob >= thresh_nn).astype(int)

    from sklearn.metrics import accuracy_score, average_precision_score
    res_nn = {
        'accuracy': accuracy_score(y_test, y_pred_nn),
        'precision': precision_score(y_test, y_pred_nn, zero_division=0),
        'recall': recall_score(y_test, y_pred_nn, zero_division=0),
        'f1': f1_score(y_test, y_pred_nn, zero_division=0),
        'auc': roc_auc_score(y_test, nn_prob),
        'avg_precision': average_precision_score(y_test, nn_prob),
        'y_pred': y_pred_nn,
        'y_prob': nn_prob,
        'y_true': y_test,
        'train_time': nn_time,
        'threshold': thresh_nn,
    }

    nn_path = os.path.join(OUTPUT_DIR, 'mlbfd_mega_neural_network_model.keras')
    nn_model.save(nn_path)

    print(f'\n  ✅ Done in {nn_time:.1f}s')
    print(f'  Precision : {res_nn["precision"]:.4f}')
    print(f'  Recall    : {res_nn["recall"]:.4f}')
    print(f'  F1-Score  : {res_nn["f1"]:.4f}')
    print(f'  ROC-AUC   : {res_nn["auc"]:.4f}')
    print(f'  Saved → {nn_path}')

except ImportError:
    print('  ⚠️  TensorFlow not installed — Neural Network skipped')
    print('  Install with: pip install tensorflow')

gc.collect()

## Cell 5: Part 3 — Ensemble Aggregation

In [ ]:
# Collect all results
all_results = {
    'XGBoost': res_xgb,
    'Random Forest': res_rf,
    'Logistic Regression': res_lr,
    'Isolation Forest': res_iso,
}
if res_nn is not None:
    all_results['Neural Network'] = res_nn

trained_models = {
    'XGBoost': xgb_model,
    'Random Forest': rf_model,
    'Logistic Regression': lr_model,
    'Isolation Forest': iso_model,
}
if nn_model is not None:
    trained_models['Neural Network'] = nn_model

# Weighted ensemble
print('Computing weighted ensemble...')
ensemble_weights = {
    'XGBoost': 0.35,
    'Random Forest': 0.25,
    'Neural Network': 0.15,
    'Logistic Regression': 0.15,
    'Isolation Forest': 0.10,
}
ensemble_res = _weighted_ensemble(all_results, weights=ensemble_weights)
all_results['Ensemble'] = ensemble_res

print(f'\nEnsemble Results:')
print(f'  Threshold  : {ensemble_res["threshold"]:.4f}')
print(f'  Precision  : {ensemble_res["precision"]:.4f}')
print(f'  Recall     : {ensemble_res["recall"]:.4f}')
print(f'  F1-Score   : {ensemble_res["f1"]:.4f}')
print(f'  ROC-AUC    : {ensemble_res["auc"]:.4f}')

## Cell 6: Part 4 — Evaluation & Visualisation

In [ ]:
# ─── Summary Table ─────────────────────────────────────────────────────
summary_data = []
for name, res in all_results.items():
    summary_data.append({
        'Model': name,
        'Accuracy': f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}",
        'Recall': f"{res['recall']:.4f}",
        'F1': f"{res['f1']:.4f}",
        'AUC': f"{res['auc']:.4f}",
        'Train Time': f"{res.get('train_time', 0):.1f}s" if name != 'Ensemble' else '—',
    })

summary_df = pd.DataFrame(summary_data)
print('Phase 11 Model Performance Summary')
print('=' * 80)
print(summary_df.to_string(index=False))
print('=' * 80)

In [ ]:
# ─── Plots ─────────────────────────────────────────────────────────────
_plot_roc(all_results, os.path.join(OUTPUT_DIR, 'mlbfd_mega_roc_curves.png'))
plt.show()

_plot_pr(all_results, os.path.join(OUTPUT_DIR, 'mlbfd_mega_precision_recall.png'))
plt.show()

_plot_confusion_matrices(all_results, os.path.join(OUTPUT_DIR, 'mlbfd_mega_confusion_matrices.png'))
plt.show()

_plot_model_comparison(
    {k: v for k, v in all_results.items() if k != 'Ensemble'},
    os.path.join(OUTPUT_DIR, 'mlbfd_mega_model_comparison.png')
)
plt.show()

_plot_xgb_importance(xgb_model, feature_names,
                     os.path.join(OUTPUT_DIR, 'mlbfd_mega_xgb_importance.png'))
plt.show()

print('✅ All plots saved to', OUTPUT_DIR)

In [ ]:
# ─── Optional SHAP Analysis ────────────────────────────────────────────
try:
    import shap
    print('Computing SHAP values for XGBoost...')
    explainer = shap.TreeExplainer(xgb_model)
    # Use a sample for speed
    sample_size = min(1000, len(X_test_sc))
    X_shap = X_test_sc[:sample_size]
    shap_values = explainer.shap_values(X_shap)

    plt.figure()
    shap.summary_plot(shap_values, X_shap,
                      feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'mlbfd_mega_shap_summary.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

    plt.figure()
    shap.summary_plot(shap_values, X_shap,
                      feature_names=feature_names, plot_type='bar', show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'mlbfd_mega_shap_bar.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ SHAP plots saved')
except ImportError:
    print('⚠️  SHAP not installed — skipped. Install with: pip install shap')

## Cell 7: Part 5 — Model Export

In [ ]:
# ─── Save sklearn/XGBoost models ─────────────────────────────────────
model_file_map = {
    'XGBoost':             'mlbfd_mega_xgboost_model.pkl',
    'Random Forest':       'mlbfd_mega_random_forest_model.pkl',
    'Logistic Regression': 'mlbfd_mega_logistic_regression_model.pkl',
    'Isolation Forest':    'mlbfd_mega_isolation_forest_model.pkl',
}
for name, fname in model_file_map.items():
    path = os.path.join(OUTPUT_DIR, fname)
    with open(path, 'wb') as f:
        pickle.dump(trained_models[name], f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  ✅ {name} → {fname} ({size_mb:.1f} MB)')

# Save results
results_save = {
    name: {k: v for k, v in res.items()
           if k not in ('y_pred', 'y_prob', 'y_true')}
    for name, res in all_results.items()
}
results_path = os.path.join(OUTPUT_DIR, 'mlbfd_mega_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(results_save, f)
print(f'  ✅ Results metrics → mlbfd_mega_results.pkl')

# Save dataset info
dataset_info = {
    'total_rows': len(X_raw),
    'total_features': len(feature_names),
    'feature_names': feature_names,
    'datasets_used': ['Phase 2B Mega (synthetic replication)'] if USE_SYNTHETIC else ['Phase 2B Mega'],
    'dataset_stats': {
        'Phase 11': {
            'rows': len(X_raw),
            'fraud': int(y.sum()),
            'fraud_pct': float(y.mean() * 100),
            'source': 'phase11',
        }
    },
    'sources': ['phase11'],
    'fraud_total': int(y.sum()),
    'fraud_pct': float(y.mean() * 100),
    'training_date': datetime.now().strftime('%Y-%m-%d'),
    'phase': 'Phase 11',
    'best_model': max(
        (n for n in all_results if n != 'Ensemble'),
        key=lambda n: all_results[n].get('auc', 0),
    ),
    'best_auc': max(
        all_results[n].get('auc', 0) for n in all_results if n != 'Ensemble'
    ),
    'models_trained': list(trained_models.keys()),
    'ensemble_threshold': ensemble_res['threshold'],
    'ensemble_f1': ensemble_res['f1'],
    'ensemble_precision': ensemble_res['precision'],
    'ensemble_recall': ensemble_res['recall'],
}
info_path = os.path.join(OUTPUT_DIR, 'mlbfd_mega_dataset_info.pkl')
with open(info_path, 'wb') as f:
    pickle.dump(dataset_info, f)
print(f'  ✅ Dataset info → mlbfd_mega_dataset_info.pkl')
print(f'\n✅ All models saved to {os.path.abspath(OUTPUT_DIR)}')

## Cell 8: Success Criteria Verification

In [ ]:
ens = all_results['Ensemble']
p, r, f1, auc = ens['precision'], ens['recall'], ens['f1'], ens['auc']

print('\n' + '='*60)
print('PHASE 11 SUCCESS CRITERIA CHECK')
print('='*60)
print(f'  Ensemble F1     : {f1:.4f}  {"✅" if f1 >= 0.92 else "❌"} (target ≥ 0.92)')
print(f'  Ensemble Prec.  : {p:.4f}  {"✅" if p >= 0.90 else "❌"} (target ≥ 0.90)')
print(f'  Ensemble Recall : {r:.4f}  {"✅" if r >= 0.85 else "❌"} (target ≥ 0.85)')
print(f'  Ensemble AUC    : {auc:.4f}  ✅')
print('='*60)

all_passed = f1 >= 0.92 and p >= 0.90 and r >= 0.85
if all_passed:
    print('\n🎉 ALL PHASE 11 SUCCESS CRITERIA MET!')
    print('   Models are ready for predictor.py integration.')
else:
    print('\n⚠️  Some criteria not met. Review hyperparameters.')

# Detailed per-model report
print('\nPer-Model Summary:')
print(f'{"Model":<22} {"Precision":>10} {"Recall":>8} {"F1":>8} {"AUC":>10}')
print('-'*62)
for name, res in all_results.items():
    print(f'{name:<22} {res["precision"]:>10.4f} {res["recall"]:>8.4f} '
          f'{res["f1"]:>8.4f} {res["auc"]:>10.4f}')